# Daniel Kalo, Pranjal Rane
# CS 6120
# Summer 2024
# Final Project - Development of a Clinical Chatbot for Patient Support Using NLP

# Part 1: Collecting and Preprocessing Data from Datasets
## Importing all necessary modules

In [1]:
import os
import pandas as pd
import xml.etree.ElementTree as ET
import nltk
import json
import re
import torch
import numpy as np
import time
import spacy
import logging
import pickle
import sys
from nltk.corpus import stopwords
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from spacy.matcher import PhraseMatcher
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, BertForQuestionAnswering, BertModel, Trainer, TrainingArguments
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/danielkalo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/danielkalo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/danielkalo/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## MedQuAD Data Collecting and Preprocessing

In [ ]:
# Function to parse XML files and extract data
def parse_xml(file_path):
    """
    Parses an XML file and extracts question-answer pairs.

    Args:
        file_path (str): The path to the XML file.

    Returns:
        pd.DataFrame: A DataFrame containing the extracted question-answer pairs.
    """
    tree = ET.parse(file_path)
    root = tree.getroot()
    data = []
    for child in root:
        if child.tag == 'QAPairs':
            for qa_pair in child:
                question_element = None
                answer_element = None
                for qa_subchild in qa_pair:
                    if qa_subchild.tag == 'Question':
                        question_element = qa_subchild
                    elif qa_subchild.tag == 'Answer':
                        answer_element = qa_subchild
                
                if question_element is not None and answer_element is not None:
                    question = question_element.text.strip() if question_element.text else ''
                    answer = answer_element.text.strip() if answer_element.text else ''
                    data.append({'question': question, 'answer': answer})
    
    if not data:
        print(f"No data found in file: {file_path}")
    return pd.DataFrame(data)

# Function to preprocess text
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Preprocesses a given text by tokenizing, removing stopwords, and lemmatizing.

    Args:
        text (str): The text to preprocess.

    Returns:
        str: The preprocessed text.
    """
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word.lower()) for word in tokens if word.isalpha() and word.lower() not in stop_words]
    return ' '.join(tokens)

In [ ]:
base_dir = 'MedQuAD'
directories = ['1_CancerGov_QA', '2_GARD_QA', '3_GHR_QA', '5_NIDDK_QA', '6_NINDS_QA', 
               '7_SeniorHealth_QA', '8_NHLBI_QA_XML', '9_CDC_QA', '10_MPlus_ADAM_QA', '11_MPlusDrugs_QA', '12_MPlusHerbsSupplements_QA']

data_frames = []
for dir_name in directories:
    dir_path = os.path.join(base_dir, dir_name)
    for file_name in os.listdir(dir_path):
        if file_name.endswith('.xml'):
            file_path = os.path.join(dir_path, file_name)
            df = parse_xml(file_path)
            if not df.empty:
                data_frames.append(df)

# Check if any data frames were collected
if not data_frames:
    raise ValueError("No valid data found in any XML files.")

combined_df = pd.concat(data_frames, ignore_index=True)

# Preprocess the questions and answers
combined_df['preprocessed_question'] = combined_df['question'].apply(preprocess_text)
combined_df['preprocessed_answer'] = combined_df['answer'].apply(preprocess_text)

combined_df.to_csv('MedQuAD_preprocessed.csv', index=False)

In [ ]:
# Load the preprocessed data
combined_df = pd.read_csv('MedQuAD_preprocessed.csv')

# Ensure NLTK resources are available
nltk.download('punkt')

# Function to normalize text
def normalize_text(text):
    """
    Normalizes a given text by removing special characters and converting to lowercase.

    Args:
        text (str): The text to normalize.

    Returns:
        str: The normalized text.
    """
    if isinstance(text, float):
        text = ""  # Handle NaN values
    # Remove special characters
    text = re.sub(r'\W', ' ', text)
    # Convert to lowercase
    text = text.lower()
    return text

# Additional preprocessing steps
combined_df.dropna(subset=['question', 'answer'], inplace=True)
combined_df.drop_duplicates(subset=['question', 'answer'], inplace=True)

# Ensure all entries are strings
combined_df['preprocessed_question'] = combined_df['preprocessed_question'].astype(str)
combined_df['preprocessed_answer'] = combined_df['preprocessed_answer'].astype(str)

combined_df['normalized_question'] = combined_df['preprocessed_question'].apply(normalize_text)
combined_df['normalized_answer'] = combined_df['preprocessed_answer'].apply(normalize_text)
combined_df['tokenized_question'] = combined_df['normalized_question'].apply(word_tokenize)
combined_df['tokenized_answer'] = combined_df['normalized_answer'].apply(word_tokenize)

# Save the final preprocessed data
combined_df.to_csv('MedQuAD_final_preprocessed.csv', index=False)

## COVID-QA Data Collecting and Preprocessing

In [ ]:
# Load the dataset
with open('COVID-QA/data/question-answering/COVID-QA.json') as f:
    data = json.load(f)

# Extract question-answer pairs based on the inspected structure
qa_pairs = []
for entry in data['data']:  # Accessing 'data' key
    for paragraph in entry['paragraphs']:
        for qa in paragraph['qas']:
            question = qa['question']
            answer = qa['answers'][0]['text'] if qa['answers'] else ''
            qa_pairs.append({'question': question, 'answer': answer})

# Convert to DataFrame
df = pd.DataFrame(qa_pairs)

# Additional preprocessing steps
df.dropna(subset=['question', 'answer'], inplace=True)
df.drop_duplicates(subset=['question', 'answer'], inplace=True)

# Ensure all entries are strings
df['question'] = df['question'].astype(str)
df['answer'] = df['answer'].astype(str)

df['normalized_question'] = df['question'].apply(normalize_text)
df['normalized_answer'] = df['answer'].apply(normalize_text)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

df['tokenized_question'] = df['normalized_question'].apply(preprocess_text)
df['tokenized_answer'] = df['normalized_answer'].apply(preprocess_text)

# Save the preprocessed data
df.to_csv('COVID_QA_preprocessed.csv', index=False)

In [ ]:
# Load the COVID-QA preprocessed data
covid_qa_df = pd.read_csv('COVID_QA_preprocessed.csv')

# Function to preprocess text (same as used for MedQuAD)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Add preprocessed_question and preprocessed_answer columns
covid_qa_df['preprocessed_question'] = covid_qa_df['question'].apply(preprocess_text)
covid_qa_df['preprocessed_answer'] = covid_qa_df['answer'].apply(preprocess_text)

# Save the updated COVID-QA dataset
covid_qa_df.to_csv('COVID_QA_final_preprocessed.csv', index=False)

# Part 2: Intent Recognition and Entity Extraction
## Commenting out the below cell, as it was only used originally to combine the final preprocessed datasets, in order to manually label their intents on 200 sample questions

In [ ]:
'''
# Load preprocessed datasets
medquad_df = pd.read_csv('MedQuAD_final_preprocessed.csv')
covid_qa_df = pd.read_csv('COVID_QA_final_preprocessed.csv')

# Combine datasets
combined_df = pd.concat([medquad_df[['question']], covid_qa_df[['question']]], ignore_index=True)

# Sample a subset of questions for initial manual labeling
sampled_questions = combined_df.sample(n=200, random_state=42)

# Save to CSV for manual labeling
sampled_questions.to_csv('initial_sampled_questions_for_labeling.csv', index=False)
'''

## Displaying labeled intents' distribution

In [ ]:
# Load initial labeled data
labeled_data = pd.read_csv('initial_sampled_questions_for_labeling.csv')

# Inspect the unique intents and their distribution
unique_intents = labeled_data['intent'].unique()
intent_distribution = labeled_data['intent'].value_counts()

print("Unique Intents:", unique_intents)
print("Intent Distribution:\n", intent_distribution)

## Prepare labeled sample data for BERT training - Predicting the remaining intents

In [3]:
# Load initial labeled data
labeled_data = pd.read_csv('initial_sampled_questions_for_labeling.csv')

# Ensure no missing values in the 'question' and 'intent' columns
labeled_data.dropna(subset=['question', 'intent'], inplace=True)

# Ensure all entries are strings
labeled_data['question'] = labeled_data['question'].astype(str)

# Map intents to numerical labels
label_to_id = {label: idx for idx, label in enumerate(labeled_data['intent'].unique())}
id_to_label = {idx: label for label, idx in label_to_id.items()}

# Add numerical labels to the DataFrame
labeled_data['intent_id'] = labeled_data['intent'].map(label_to_id)

In [10]:
# Separate each class
intent_dataframes = [labeled_data[labeled_data['intent'] == intent] for intent in labeled_data['intent'].unique()]

# Find the maximum class size
max_class_size = max([len(df) for df in intent_dataframes])

# Resample each class to have the same size
balanced_dataframes = [resample(df, replace=True, n_samples=max_class_size, random_state=42) for df in intent_dataframes]

# Combine balanced dataframes
balanced_labeled_data = pd.concat(balanced_dataframes)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(balanced_labeled_data['question'], balanced_labeled_data['intent_id'], test_size=0.2, random_state=42)

# Tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(label_to_id))

# Tokenize data
train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=128)

# Create dataset objects
class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IntentDataset(train_encodings, y_train.tolist())
test_dataset = IntentDataset(test_encodings, y_test.tolist())

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',          
    num_train_epochs=20,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=64,   
    warmup_steps=500,                
    weight_decay=0.01,               
    logging_dir='./logs',            
    logging_steps=10,
)

# Trainer
trainer = Trainer(
    model=model,                        
    args=training_args,                  
    train_dataset=train_dataset,         
    eval_dataset=test_dataset            
)

# Train the model
trainer.train()

# Evaluate the model
trainer.evaluate()

# Save the model
trainer.save_model('./results')

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

Step,Training Loss
10,1.992200
20,2.019800
30,1.926100
40,1.822900
50,1.773300
60,1.681200
70,1.561600
80,1.428700
90,1.404000
100,1.268800


In [5]:
# Load the entire preprocessed dataset
medquad_df = pd.read_csv('MedQuAD_final_preprocessed.csv')
covid_qa_df = pd.read_csv('COVID_QA_final_preprocessed.csv')

# Combine both datasets if needed
full_df = pd.concat([medquad_df, covid_qa_df], ignore_index=True)

# Ensure no missing values in the 'question' column
full_df.dropna(subset=['question'], inplace=True)

# Ensure all entries are strings
full_df['question'] = full_df['question'].astype(str)

# Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the full dataset
full_encodings = tokenizer(list(full_df['question']), truncation=True, padding=True, max_length=128)

# Create dataset object for full data
class FullDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

full_dataset = FullDataset(full_encodings)

# Load the trained model
model = BertForSequenceClassification.from_pretrained('./results')

# Trainer
trainer = Trainer(model=model)

# Predict the intents for the full dataset
predictions = trainer.predict(full_dataset)

# Get the predicted labels
predicted_labels = torch.argmax(torch.tensor(predictions.predictions), dim=1).numpy()

# Map numerical labels back to string intents
id_to_label = {idx: label for label, idx in label_to_id.items()}
full_df['predicted_intent'] = [id_to_label[label] for label in predicted_labels]

# Save the dataset with predicted intents
full_df.to_csv('full_dataset_with_predicted_intents.csv', index=False)

In [6]:
# Display intents' distribution
predicted_intents_distribution = full_df['predicted_intent'].value_counts()
print(predicted_intents_distribution)

predicted_intent
 Condition_Information    5505
 Treatment_Information    3061
 Symptom_Checking         2770
 Genetic_Information      2607
 Research_Updates         2252
 Diagnosis_Information     846
 Risk_Factors              356
Name: count, dtype: int64


## Evaluating Precision, Recall, F1-Score, and Support metrics for trained BERT intent-labeling model

In [11]:
# Evaluate the model
results = trainer.evaluate()

# Predict the intents for the test dataset
predictions = trainer.predict(test_dataset)
predicted_labels = torch.argmax(torch.tensor(predictions.predictions), dim=1).numpy()

# Map numerical labels back to string intents for test data
y_test_labels = [id_to_label[label] for label in y_test]
predicted_test_labels = [id_to_label[label] for label in predicted_labels]

# Generate classification report
print(classification_report(y_test_labels, predicted_test_labels, target_names=id_to_label.values()))

                        precision    recall  f1-score   support

   Genetic_Information       1.00      1.00      1.00        16
 Condition_Information       0.94      1.00      0.97        17
      Symptom_Checking       1.00      1.00      1.00        15
      Research_Updates       1.00      1.00      1.00         8
 Treatment_Information       1.00      1.00      1.00        11
          Risk_Factors       1.00      1.00      1.00        14
 Diagnosis_Information       1.00      0.92      0.96        12

              accuracy                           0.99        93
             macro avg       0.99      0.99      0.99        93
          weighted avg       0.99      0.99      0.99        93



## Extracting entities with SpaCy and saving them into a csv file - this file is mainly used going forward

In [ ]:
# Load the full dataset with predicted intents
full_df = pd.read_csv('full_dataset_with_predicted_intents.csv')

# Load a pre-trained SpaCy model
nlp = spacy.load('en_core_web_sm')

# Custom function to refine entity extraction
def extract_entities(text):
    """
    Extracts entities using SpaCy's NER and custom rules.

    Args:
        text (str): The text from which to extract entities.

    Returns:
        List[Tuple[str, str]]: A list of tuples containing the entity text and its label.
    """
    doc = nlp(text)
    entities = [(entity.text, entity.label_) for entity in doc.ents]

    # Custom rules for entity extraction, e.g., capturing medical terms or specific patterns
    custom_patterns = {
        'DISEASE': r'\b(cancer|diabetes|hypertension|asthma|COVID-19)\b',
        'MEDICATION': r'\b(aspirin|ibuprofen|metformin|lisinopril)\b',
        'SYMPTOM': r'\b(headache|fever|cough|pain|nausea)\b'
    }

    # Apply custom patterns
    for label, pattern in custom_patterns.items():
        matches = re.finditer(pattern, text, re.IGNORECASE)
        for match in matches:
            entities.append((match.group(), label))

    return entities

# Extract entities for each question
full_df['entities'] = full_df['question'].apply(extract_entities)

# Save the dataset with predicted intents and extracted entities
full_df.to_csv('full_dataset_with_intents_and_entities_2.0.csv', index=False)

# Example usage:
text_example = "The patient has a history of hypertension and is currently taking lisinopril."
entities = extract_entities(text_example)
print(entities)

# Part 3: Response Generation
## Training Linear Regression and Random Forest models on a sample of half the data for response generation
## NOTE: 'full_dataset_with_intents_and_entities_2.0_copy.csv' is based on the above 'full_dataset_with_intents_and_entities_2.0.csv' file. The copy version has duplicates and blanks removed for a more concise and accurate dataset

In [ ]:
# LinearRegression, RandomForestRegressor

# Load the full dataset with predicted intents and entities
full_df = pd.read_csv('full_dataset_with_intents_and_entities_2.0_copy.csv')

# Ensure no NaN values in any relevant columns
relevant_columns = ['question', 'answer', 'preprocessed_question', 'preprocessed_answer', 'normalized_question', 'normalized_answer', 'tokenized_question', 'tokenized_answer', 'predicted_intent', 'entities']
full_df.dropna(subset=relevant_columns, inplace=True)

# Sample questions from the full dataset
sampled_df = full_df.sample(n=3500, random_state=42)

# Combine preprocessed question, intents, and entities into a single feature set
sampled_df['combined_features'] = sampled_df.apply(lambda row: f"{row['preprocessed_question']} {row['predicted_intent']} {' '.join([entity[0] for entity in eval(row['entities'])])}", axis=1)

# Prepare the data
X = sampled_df['combined_features']
y = sampled_df['preprocessed_answer']

# Split data into training and evaluation sets
X_train, X_eval, y_train, y_eval = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize TF-IDF vectorizer for X (questions) and y (answers)
vectorizer_X = TfidfVectorizer()
vectorizer_y = TfidfVectorizer()

# Fit and transform the training data for X
print("Fitting and transforming the training data...")
start_time = time.time()
X_train_tfidf = vectorizer_X.fit_transform(X_train)
end_time = time.time()
print(f"Fitting and transforming completed in {end_time - start_time:.2f} seconds")

# Transform the evaluation data for X
print("Transforming the evaluation data...")
start_time = time.time()
X_eval_tfidf = vectorizer_X.transform(X_eval)
end_time = time.time()
print(f"Transforming completed in {end_time - start_time:.2f} seconds")

# Fit and transform the training data for y
print("Fitting and transforming the training data for y...")
start_time = time.time()
y_train_tfidf = vectorizer_y.fit_transform(y_train)
end_time = time.time()
print(f"Fitting and transforming completed in {end_time - start_time:.2f} seconds")

# Transform the evaluation data for y
print("Transforming the evaluation data for y...")
start_time = time.time()
y_eval_tfidf = vectorizer_y.transform(y_eval)
end_time = time.time()
print(f"Transforming completed in {end_time - start_time:.2f} seconds")

# Convert sparse matrices to dense format
X_train_tfidf = X_train_tfidf.toarray()
y_train_tfidf = y_train_tfidf.toarray()
X_eval_tfidf = X_eval_tfidf.toarray()
y_eval_tfidf = y_eval_tfidf.toarray()

# Initialize and train the Linear Regression model
print("Training the Linear Regression model...")
start_time = time.time()
linear_regression_model = LinearRegression()
linear_regression_model.fit(X_train_tfidf, y_train_tfidf)
end_time = time.time()
print(f"Linear Regression model training completed in {end_time - start_time:.2f} seconds")

# Random Forest Regressor Model
print("Training the Random Forest Regressor model...")
start_time = time.time()
random_forest_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
random_forest_model.fit(X_train_tfidf, y_train_tfidf)
end_time = time.time()
print(f"Random Forest Regressor model training completed in {end_time - start_time:.2f} seconds")

## Verifying usefulness of Linear Regression model - NOT USED GOING FORWARD DUE TO RESULTS

In [ ]:
# Generate responses for the evaluation set using Linear Regression model
print("Generating responses with Linear Regression model...")
start_time = time.time()
y_pred_lr = linear_regression_model.predict(X_eval_tfidf)
end_time = time.time()
print(f"Response generation with Linear Regression completed in {end_time - start_time:.2f} seconds")

# Evaluate the Linear Regression model using Mean Squared Error
mse_lr = mean_squared_error(y_eval_tfidf, y_pred_lr)
print(f"Linear Regression Mean Squared Error: {mse_lr}")

# Convert the generated responses back to text
y_pred_lr_text = vectorizer_y.inverse_transform(y_pred_lr)

# Save generated responses
generated_responses_lr = pd.DataFrame({'question': X_eval, 'generated_response': [' '.join(text) for text in y_pred_lr_text]})
generated_responses_lr.to_csv('generated_responses_linear_regression.csv', index=False)

# Display a sample response
print(f"Linear Regression Generated Response: {' '.join(y_pred_lr_text[0])}")

## Verifying usefulness of Random Forest model - NOT USED GOING FORWARD DUE TO RESULTS

In [ ]:
# Generate responses for the evaluation set using Random Forest Regressor model
print("Generating responses with Random Forest Regressor model...")
start_time = time.time()
y_pred_rf = random_forest_model.predict(X_eval_tfidf)
end_time = time.time()
print(f"Response generation with Random Forest Regressor completed in {end_time - start_time:.2f} seconds")

# Evaluate the Random Forest Regressor model using Mean Squared Error
mse_rf = mean_squared_error(y_eval_tfidf, y_pred_rf)
print(f"Random Forest Regressor Mean Squared Error: {mse_rf}")

# Convert the generated responses back to text
y_pred_rf_text = vectorizer_y.inverse_transform(y_pred_rf)

# Save generated responses
generated_responses_rf = pd.DataFrame({'question': X_eval, 'generated_response': [' '.join(text) for text in y_pred_rf_text]})
generated_responses_rf.to_csv('generated_responses_random_forest.csv', index=False)

# Display a sample response
print(f"Random Forest Generated Response: {' '.join(y_pred_rf_text[0])}")

## Random Forest model was more accurate than the Linear Regression model, so this is a function to generate responses based on this model and TF-IDF vectorization - DID NOT USE

In [ ]:
# Load the saved BERT model and tokenizer
intent_model = BertForSequenceClassification.from_pretrained('./results')
intent_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Map numerical labels back to string intents (use the same mapping you created earlier)
id_to_label = {idx: label for label, idx in label_to_id.items()}

# Function to generate a response given an input question
def generate_response_bert_tfidf(input_question, model, vectorizer_X, vectorizer_y, intent_model, intent_tokenizer, entity_extractor):
    """
    Generates a response to an input question using a BERT-based intent classifier 
    and TF-IDF features.

    Args:
        input_question (str): The input question from which to generate a response.
        model: The machine learning model trained to predict responses based on TF-IDF features.
        vectorizer_X: The TF-IDF vectorizer used to transform input features.
        vectorizer_y: The TF-IDF vectorizer used to transform output responses.
        intent_model: The BERT-based model used to predict the intent of the input question.
        intent_tokenizer: The tokenizer associated with the BERT intent model.
        entity_extractor (function): A function to extract entities from the input question.

    Returns:
        str: The generated response as a string.
    """
    # Preprocess the input question similarly to how the training data was preprocessed
    preprocessed_question = preprocess_text(input_question)
    
    # Predict the intent of the input question
    intent_tfidf = intent_tokenizer(preprocessed_question, truncation=True, padding=True, max_length=128, return_tensors='pt')
    predicted_intent_id = intent_model(intent_tfidf['input_ids']).logits.argmax(dim=1).item()
    predicted_intent = id_to_label[predicted_intent_id]
    
    # Extract entities from the input question
    entities = entity_extractor(preprocessed_question)
    entity_text = ' '.join([entity[0] for entity in entities])
    
    # Combine preprocessed question, predicted intent, and entities into a feature string
    combined_features = f"{preprocessed_question} {predicted_intent} {entity_text}"
    
    # Transform the combined features into TF-IDF features
    input_tfidf = vectorizer_X.transform([combined_features]).toarray()
    
    # Generate a response using the model
    generated_response_tfidf = model.predict(input_tfidf)
    
    # Convert the response back to text
    generated_response_text = vectorizer_y.inverse_transform(generated_response_tfidf)
    return ' '.join(generated_response_text[0])

# Example usage with updated generate_response function

# Input a sample question
input_question = "When did the World Health Organization (WHO) officially declare the 2019-nCoV epidemic as a Public Health Emergency of International Concern?"

# Generate and print the response
response = generate_response_bert_tfidf(input_question, random_forest_model, vectorizer_X, vectorizer_y, intent_model, intent_tokenizer, extract_entities)
print(f"Generated Response: {response}")

## Trying to obtain better responses with only a pre-trained BERT model and no TF-IDF vectorization - DID NOT USE

In [ ]:
# Load the full dataset with predicted intents and entities
full_df = pd.read_csv('full_dataset_with_intents_and_entities_2.0_copy.csv')

# Ensure no NaN values in any relevant columns
relevant_columns = [
    'question', 'answer', 'preprocessed_question', 'preprocessed_answer', 
    'normalized_question', 'normalized_answer', 'tokenized_question', 
    'tokenized_answer', 'predicted_intent', 'entities'
]
full_df.dropna(subset=relevant_columns, inplace=True)

# Sample questions from the full dataset
sampled_df = full_df.sample(n=1000, random_state=42)

# Ensure all entries are strings
sampled_df['question'] = sampled_df['question'].astype(str)
sampled_df['answer'] = sampled_df['answer'].astype(str)

# Load the BERT tokenizer and model for question answering
qa_model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')
qa_tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
qa_model.to(device)

# Function to generate responses
def generate_responses_bert_v2(questions, contexts, qa_model, qa_tokenizer, max_length=512, num_samples=None):
    """
    Generates responses to a list of questions based on provided contexts using a BERT-based QA model.

    Args:
        questions (List[str]): A list of questions to be answered.
        contexts (List[str]): A list of contexts corresponding to each question.
        qa_model: The BERT-based QA model used to generate responses.
        qa_tokenizer: The tokenizer associated with the BERT-based QA model.
        max_length (int, optional): The maximum length for tokenizing inputs. Defaults to 512.
        num_samples (int, optional): If specified, limits the number of questions and contexts processed. Defaults to None.

    Returns:
        List[str]: A list of responses generated for each question based on the corresponding context.
    """
    responses = []
    
    if num_samples:
        questions = questions[:num_samples]
        contexts = contexts[:num_samples]
    
    for question, context in zip(questions, contexts):
        if not context:  # Skip empty contexts
            continue
        
        # Tokenize the input with a suitable truncation strategy
        inputs = qa_tokenizer(
            question, context,
            add_special_tokens=True,
            truncation=True,
            max_length=max_length,
            padding='max_length',  # Ensure uniform input size
            return_tensors='pt'
        )
        input_ids = inputs['input_ids'].to(device)
        attention_mask = inputs['attention_mask'].to(device)
        token_type_ids = inputs['token_type_ids'].to(device)
        
        # Get the model output
        outputs = qa_model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        start_scores, end_scores = outputs.start_logits, outputs.end_logits
        
        # Decode the answer
        start_index = torch.argmax(start_scores)
        end_index = torch.argmax(end_scores) + 1
        answer = qa_tokenizer.convert_tokens_to_string(qa_tokenizer.convert_ids_to_tokens(input_ids[0][start_index:end_index]))
        
        if not answer.strip() or '[CLS]' in answer or '[SEP]' in answer:
            answer = "I'm sorry, I couldn't find a relevant answer."
        
        responses.append(answer)
    
    return responses

# Batch processing
batch_size = 25
all_responses = []

for start_idx in range(0, len(sampled_df), batch_size):
    end_idx = min(start_idx + batch_size, len(sampled_df))
    batch_questions = sampled_df['question'][start_idx:end_idx].tolist()
    batch_contexts = sampled_df['answer'][start_idx:end_idx].tolist()
    
    # Generate responses for the batch with intents
    batch_responses = generate_responses_bert_v2(batch_questions, batch_contexts, qa_model, qa_tokenizer)
    all_responses.extend(batch_responses)

# Add generated responses to the dataframe
sampled_df['generated_response'] = all_responses

# Save the dataset with generated responses
sampled_df.to_csv('full_dataset_with_generated_responses.csv', index=False)

# Display a few samples
print(sampled_df[['question', 'predicted_intent', 'generated_response']].head())

## ***THIS MODEL IS THE ONE WE USED***
## Trained a GPT-2 model based on the full dataset ('full_dataset_with_intents_and_entities_2.0_copy.csv'), yet only focused on the raw (non-preprocessed) questions and answers, as in the end, they generated more accurate and valid responses
## 2:30 hours of training time!

In [ ]:
# Load pre-trained model and tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Set pad_token_id to eos_token_id to avoid errors
tokenizer.pad_token = tokenizer.eos_token

# Load dataset
full_df = pd.read_csv('full_dataset_with_intents_and_entities_2.0_copy.csv')

# Ensure no NaN values in any relevant columns
relevant_columns = ['question', 'answer']
full_df.dropna(subset=relevant_columns, inplace=True)

# Prepare text inputs for fine-tuning
# Focus purely on the answer, using a special marker to indicate where the answer begins
full_df['input_text'] = full_df.apply(
    lambda row: f"{row['answer']}", axis=1
)

class CustomDataset(Dataset):
    def __init__(self, texts, tokenizer, block_size=128):
        self.examples = []
        for text in texts:
            tokens = tokenizer(text, truncation=True, padding='max_length', max_length=block_size, return_tensors="pt")
            self.examples.append(tokens.input_ids.squeeze())

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return {"input_ids": self.examples[i]}

# Create the dataset
train_dataset = CustomDataset(full_df['input_text'].tolist(), tokenizer)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# Fine-tuning arguments
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    save_steps=1000,
    save_total_limit=2,
    prediction_loss_only=True,
    logging_steps=200,
    logging_dir="./logs",
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

# Train the model
trainer.train()

# Save the model
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

# Pickle the trained model for easy reuse
with open('gpt2_finetuned_model.pkl', 'wb') as f:
    pickle.dump(model, f)

## Loading pickled GPT-2 model to integrate into a response generation function

In [ ]:
# Reload the model from the pickle file
with open('gpt2_finetuned_model.pkl', 'rb') as f:
    model = pickle.load(f)

# Ensure the model is on the CPU
model.to("cpu")

# Reload the tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-finetuned")
tokenizer.pad_token = tokenizer.eos_token  # Set pad_token to eos_token to avoid errors

def clean_generated_response(response):
    """
    Cleans the generated response by removing any leading content up to and including the first question mark,
    and ensuring the response ends with proper sentence punctuation.

    Args:
        response (str): The generated response to be cleaned.

    Returns:
        str: The cleaned response, with any unwanted content removed and proper sentence punctuation ensured.
    """
    # If there's a question mark, remove everything up to and including it
    if '?' in response:
        response = response.split('?', 1)[-1].strip()

    # Ensure the response ends with a proper sentence punctuation
    last_punctuation = max(response.rfind('.'), response.rfind('!'), response.rfind('?'))
    if last_punctuation != -1:
        response = response[:last_punctuation + 1].strip()
    
    return response

def generate_response_gpt2(prompt):
    """
    Generates a response to the given prompt using a fine-tuned GPT-2 model and cleans the output.

    Args:
        prompt (str): The input prompt for which the response is to be generated.

    Returns:
        str: The cleaned response generated by the GPT-2 model.
    """
    inputs = tokenizer.encode(f"A: {prompt}", return_tensors="pt").to("cpu")
    attention_mask = inputs.ne(tokenizer.pad_token_id).long().to("cpu")
    outputs = model.generate(
        inputs,
        attention_mask=attention_mask,
        max_length=200,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.2
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return clean_generated_response(response)

# Example usage:
prompt = "What does obesity do to the body?"
response = generate_response_gpt2(prompt)
print(response)

# Part 4: Dialogue Management
## Integrating Daniel's GPT-2 model's response generation with Pranjal's Disease and Drug models for complete 'chat' function

In [14]:
with open('disease_model.pkl', 'rb') as f:
    disease_model = pickle.load(f)

with open('label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

with open('drug_recommendation_model.pkl', 'rb') as f:
    drug_model = pickle.load(f)

with open('tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)

# Reload the model from the pickle file
with open('gpt2_finetuned_model.pkl', 'rb') as f:
    gpt2_model = pickle.load(f)
    gpt2_model.to("cpu")

feature_names = ['Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Age', 
                 'Gender_female', 'Gender_male', 
                 'Blood Pressure_high', 'Blood Pressure_low', 'Blood Pressure_normal',
                 'Cholesterol Level_high', 'Cholesterol Level_low', 'Cholesterol Level_normal']

features = [feature.lower() for feature in feature_names]

def recommend_drugs(query):
    """
    Recommends drugs based on a user's query by finding the closest matches in a preprocessed drug review dataset.

    Args:
        query (str): The input query containing symptoms or drug-related information.

    Returns:
        Union[str, List[str]]: A list of recommended drug names based on the query, or an error message if no valid recommendations are found.
    """
    data = pd.read_csv("Cleaned_Drug_Review_Dataset_Train.csv")
    query_vec = tfidf.transform([query])
    
    try:
        distances, indices = drug_model.kneighbors(query_vec, n_neighbors=5)

        valid_indices = [i for i in indices[0] if i < len(data)]
        
        if not valid_indices:
            return "No valid drug recommendations could be found based on your query."

        recommended_drugs = data.iloc[valid_indices]['drugName'].tolist()
        
        if not recommended_drugs:
            return "No valid drug recommendations could be found based on your query."
        
        return recommended_drugs
    
    except ValueError as ve:
        return f"Error in recommendation process: {str(ve)}"
    except IndexError as ie:
        return f"Index error in recommendation process: {str(ie)}"

def predict_disease(symptoms, demographics):
    """
    Predicts a possible disease based on the provided symptoms and demographic information.

    Args:
        symptoms (List[str]): A list of symptoms reported by the user.
        demographics (Dict[str, Union[str, int]]): A dictionary containing demographic information such as age and gender.

    Returns:
        str: The predicted disease based on the input symptoms and demographics.
    """
    input_data = np.zeros(len(feature_names))

    symptom_map = {'fever': 0, 'cough': 1, 'fatigue': 2, "tired": 2, 'difficulty breathing': 3, 
                   "high blood pressure" : 7, "low blood pressure" : 8, "normal blood pressure" : 9, 
                   "low cholesterol" : 11, "high cholesterol" : 10, "normal cholesterol": 12}
    for symptom in symptoms:
        if symptom in symptom_map:
            input_data[symptom_map[symptom]] = 1

    if demographics.get("Age") is not None:
        input_data[4] = demographics.get("Age")
    
    if demographics.get("Gender") == "female":
        input_data[5] = 1
    elif demographics.get("Gender") == "male":
        input_data[6] = 1

    prediction = disease_model.predict([input_data])
    disease = label_encoder.inverse_transform(prediction)
    
    return disease[0]

def extract_entitiess(text):
    """
    Extracts symptoms and demographic information from the input text using SpaCy's NLP pipeline.

    Args:
        text (str): The input text from which to extract entities.

    Returns:
        Tuple[List[str], Dict[str, Union[str, None]]]: A tuple containing a list of extracted symptoms and a dictionary of demographic information (age and gender).
    """
    nlp = spacy.load("en_core_web_lg")
    doc = nlp(text)

    matcher = PhraseMatcher(nlp.vocab)
    
    symptoms = ["fever", "cough", "tired", "fatigue", "difficulty breathing", 
                "high blood pressure", "low blood pressure", "normal blood pressure", 
                "low cholesterol", "high cholesterol", "normal cholesterol"]
    patterns = [nlp.make_doc(symptom.lower()) for symptom in symptoms]
    matcher.add("SYMPTOM", patterns)
    
    matches = matcher(doc)
    extracted_symptoms = []
    for match_id, start, end in matches:
        extracted_symptoms.append(doc[start:end].text)
    
    demographics = {
        "Age": None,
        "Gender": None
    }
    
    for ent in doc.ents:
        if ent.label_ in ["CARDINAL", "QUANTITY", "DATE"]:
            if "year" in ent.text or ent.text.isdigit() and int(ent.text) < 120:  # Assuming ages < 120
                demographics["Age"] = ent.text
        
        gender_keywords = {"male", "female", "man", "woman", "boy", "girl"}
        for token in doc:
            if token.text.lower() in gender_keywords:
                demographics["Gender"] = token.text.lower()
    
    return extracted_symptoms, demographics

def chat(user_input, context):
    """
    Processes user input to update context with symptoms and demographics, and generates a response based on the user's intent.

    Args:
        user_input (str): The user's input query or statement.
        context (Dict[str, Union[List[str], Dict[str, Union[str, None]]]]): The context dictionary containing accumulated symptoms and demographics.

    Returns:
        Dict[str, str]: A dictionary containing the generated response.
    """
    if not user_input:
        return {"response": "I'm sorry, I didn't understand that."}

    symptoms, demographics = extract_entitiess(user_input)
    context["symptoms"].extend(symptoms)
    context["demographics"].update(demographics)

    if "diagnose" in user_input.lower() or "symptoms" in user_input.lower():
        disease = predict_disease(context["symptoms"], context["demographics"])
        response = f"Based on the symptoms you provided, you may have {disease}. Please consult with a healthcare provider for a professional diagnosis."

    elif "recommend" in user_input.lower() or "drug" in user_input.lower():
        drugs = recommend_drugs(user_input)
        response = f"Here are some drugs that may be helpful: {', '.join(drugs)}."

    else:
        response = generate_response_gpt2(user_input)

    return {"response": response}

if __name__ == "__main__":
    context = {
        "symptoms": [],
        "demographics": {},
        "condition": None,
    }

    user_input = "What does obesity do to the body?"
    result = chat(user_input, context)
    print(result["response"])

Obesity is the most common risk factor for diabetes mellitus (DM). Atypical forms of DM are characterized by a lack or shortening period between meals and dehydration, which can lead individuals with this condition. Type 1DM often appears at an early age, but some cases may be caused in childhood. In type 2DM it occurs more frequently than normal; however these types usually occur in adolescence as well -- both types have been found on a large scale. Type 3D has also occurred twice along a continuum -- once before being diagnosed during pregnancy followed shortly thereafterby another disease; now only after diagnosis. Most people who develop type 4DM end up developing insulin resistance and/or heart failure, which can lead themto high blood pressure or even death. This disorder is characterized mostly By Hirschsprung's protein deficiency  - Diabetes mellitus is a form when the pancreas doesn't get enough glucose from food.


# Part 5: Evaluation and User Testing

In [ ]:
# PRANJAL: CAN YOU ADD YOUR DATASET BELOW TO EVALUATE THE DISEASE MODEL?

from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

# Assuming you have a dataset with features and labels
X = [...]  # Your features (e.g., symptoms and demographics)
y = [...]  # Your labels (e.g., actual diseases)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
disease_model.fit(X_train, y_train)
y_pred = disease_model.predict(X_test)

precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Disease Prediction - Precision: {precision}, Recall: {recall}, F1-Score: {f1}")

In [ ]:
# PRANJAL: CAN YOU ADD YOUR DATASET BELOW TO EVALUATE THE DRUG MODEL?

# Assuming you have a test dataset with queries and expected drug recommendations
test_queries = [...]  # List of user queries
expected_drugs = [...]  # List of expected drugs for each query

y_true = []
y_pred = []

for query, expected in zip(test_queries, expected_drugs):
    recommended_drugs = recommend_drugs(query)
    y_true.append(expected)
    y_pred.append(recommended_drugs)

# Flatten lists if there are multiple correct drugs per query
y_true_flat = [drug for sublist in y_true for drug in sublist]
y_pred_flat = [drug for sublist in y_pred for drug in sublist]

precision = precision_score(y_true_flat, y_pred_flat, average='weighted')
recall = recall_score(y_true_flat, y_pred_flat, average='weighted')
f1 = f1_score(y_true_flat, y_pred_flat, average='weighted')

print(f"Drug Recommendation - Precision: {precision}, Recall: {recall}, F1-Score: {f1}")